# Grover Search Algorithm

The classical Grover search algorithm addresses the task of identifying a marked element $z$ among $2^n$ possibilities.\
The algorithm acheives query complexity $\mathcal{O}(1/\sqrt{2^n})$.\
The QAOA formulation involves the problem Hamiltonian and the standard $X$ mixer: $$H_C = |z \rangle \langle z|,\ \ \ \ H_M = \sum_{j=1}^{n}X_j.$$

In [1]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp

import time
import itertools as it
from typing import Dict, List, Tuple

In [32]:
# QAOA Hamiltonians (mixer)
def x_mixer(wires):
    """Return the standard X-mixer, H = sum_i X_i."""
    return sum(qml.PauliX(w) for w in wires)

## Case 1: $z\in \{0,1\}^{\otimes n}$ is known.

In [34]:
# QAOA Hamiltonian (cost)
# For the case when marked bitstring z is known.
def grover_oracle(beta, z_bits, wires):
    """
    beta    : phase angle
    z_bits  : list of 0/1 bits (length n)
    wires   : PennyLane wires (iterable)
    """
    # Map |z> -> |11...1>
    for bit, w in zip(z_bits, wires):
        if bit == 0:
            qml.PauliX(w)
    # Apply controlled PhaseShift
    qml.ctrl(qml.PhaseShift, control=wires[:-1])(beta, wires=wires[-1])
    # Undo mapping
    for bit, w in zip(z_bits, wires):
        if bit == 0:
            qml.PauliX(w)

In [11]:
#Uniform random bitsring from 2**n choices
n_qubits = 5
bitstring = pnp.random.randint(0, 2, n_qubits)
print(f"Random bitstring:\n{bitstring}")

Random bitstring:
[0 1 0 0 1]


In [13]:
# Find the index of the state vector 
dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev)
def bitstring_to_statevector(bitstring, n_qubits):
    qml.BasisState(bitstring, wires=list(range(n_qubits)))
    return qml.state()

state = bitstring_to_statevector(bitstring, n_qubits)
print(f"State Vector:\n{state}")
print(f"Active index: {np.argmax(np.abs(state)**2)} of {2**n_qubits-1}")

State Vector:
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j
 0.+0.j 0.+0.j]
Active index: 9 of 31


### Checkpoint - Grover mixer
Let $n$ be the number of qubits, $d= 2^n$.\
For some marked state vector $|z\rangle = |z_1z_2\cdots z_n\rangle$ with $z\in \{0,1\}^{\otimes n}$\
interpret $z$ as the index $k\in \{0,1,\ldots, 2^n -1\}$.\
Define the projector $P_z = |z\rangle \langle z|$\
$\Longrightarrow U_G(\beta)= I_d - (1 - e^{-i\beta})P_z$,\
This gives eigenvalue $e^{-i\beta}$ on vectors in $\mathbb{C}(|z\rangle)$ and $1$ on its orthogonal compliment.

In [36]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)
@qml.qnode(dev)
def grovers_circuit(beta, gamma, z_bits, n_wires):
    wires = list(range(n_wires))
    # initial state |++...+>
    for w in wires: 
       qml.Hadamard(w)
    #Apply cost Hamiltonian
    grover_oracle(beta, z_bits, wires)
    return qml.state()

In [23]:
beta = np.pi #should return phase (-1) on marked index
gamma = 0.1
bitstring = [0, 0, 0, 1]
state = grovers_circuit(beta, gamma, bitstring, n_qubits)
print(state)

[ 0.25+0.000000e+00j -0.25+3.061617e-17j  0.25+0.000000e+00j
  0.25+0.000000e+00j  0.25+0.000000e+00j  0.25+0.000000e+00j
  0.25+0.000000e+00j  0.25+0.000000e+00j  0.25+0.000000e+00j
  0.25+0.000000e+00j  0.25+0.000000e+00j  0.25+0.000000e+00j
  0.25+0.000000e+00j  0.25+0.000000e+00j  0.25+0.000000e+00j
  0.25+0.000000e+00j]


### QAOA circuit for Grover Search

In [70]:
def qaoa_layer(gamma, beta, z_bit, wires):
    # Apply grover oracle
    grover_oracle(gamma, z_bit, wires)
    # Apply mixer
    qml.ApproxTimeEvolution(x_mixer(wires), beta, 1)

Note: ```gamma``` plays the role of the Grover oracle angle\
and ```beta``` is the mixer rotation

In [72]:
def qaoa_circuit(gammas, betas, z_bit, n_wires):
    wires = list(range(n_wires))

    # Initial state |+>^n
    for w in wires:
        qml.Hadamard(w)

    for gamma, beta in zip(gammas, betas):
        qaoa_layer(gamma, beta, z_bit, wires)

#### Benchmarking QNode

Success probability vs depth $p$.\
$P_{succ}(p) = |\langle z | \psi_{p}\rangle |^2$, where $z\in \{0,1\}^{n}$ is the marked element, and\
$| \psi_{p} \rangle$ is the output state of the QAOA circuit at depth $p$. 

In [48]:
def bitstring_to_index(z_bit):
    return int("".join(str(b) for b in z_bit), 2)

In [50]:
def make_success_qnode(n_wires, z_bit):
    dev = qml.device("default.qubit", wires=n_wires)

    z_index = bitstring_to_index(z_bit)

    @qml.qnode(dev)
    def circuit(gammas, betas):
        qaoa_circuit(gammas, betas, z_bit, n_wires)
        return qml.probs()

    def success_probability(gammas, betas):
        probs = circuit(gammas, betas)
        return probs[z_index]

    return success_probability

In [90]:
# main function
def benchmark_depth(
    n_wires,
    z_bit,
    max_p,
    gamma_init=np.pi,
    beta_init=np.pi / 4,
):
    success_fn = make_success_qnode(n_wires, z_bit)

    results = {}

    for p in range(1, max_p + 1):
        gammas = np.full(p, gamma_init)
        betas = np.full(p, beta_init)
        # fixed angles for Grover's search
        P = success_fn(gammas, betas)
        results[p] = np.array(P)

    return results


### Benchmark - Checkpoint

In [66]:
def grover_optimal_iterations(n_wires):
    N = 2**n_wires
    return int(np.floor(np.pi / 4 * np.sqrt(N)))

In [92]:
n = 8
z_bit = [0,1,1,0,0,1,0,1]

data = benchmark_depth(n, z_bit, max_p=30)

p_opt = grover_optimal_iterations(n)
print("Grover optimal p:", p_opt)

Grover optimal p: 12


In [98]:
for p in data: 
    print(f"Probability of success: \n{data[p]} at depth {p}")

Probability of success: 
0.002990722656249988 at depth 1
Probability of success: 
0.0030984878540038863 at depth 2
Probability of success: 
0.002278104424476603 at depth 3
Probability of success: 
0.005670409882441101 at depth 4
Probability of success: 
0.002728259369177936 at depth 5
Probability of success: 
0.003036137418405307 at depth 6
Probability of success: 
0.0009877872051200312 at depth 7
Probability of success: 
0.007354752229046989 at depth 8
Probability of success: 
0.0023425992359844047 at depth 9
Probability of success: 
0.002806852939267346 at depth 10
Probability of success: 
0.000198433230411466 at depth 11
Probability of success: 
0.008755481864121058 at depth 12
Probability of success: 
0.0018771883273811724 at depth 13
Probability of success: 
0.002432931311485025 at depth 14
Probability of success: 
1.4252909997565025e-05 at depth 15
Probability of success: 
0.009705133403597115 at depth 16
Probability of success: 
0.0013821885307344533 at depth 17
Probability of s